# Modality Ablation — reportMerges the results from **part 1** and **part 2** into the complete8-configuration table, the deltas against the cheap GPS + Radar baseline, thescenario comparison, the plots and the interpretation.No GPU, no training, no repository clone — it reads only the per-configurationJSON files the two training notebooks produce, so it runs on Kaggle **or**locally in a few seconds.## Why this is a separate notebookThe experiment is framed around one question: what does each expensive modalitybuy *on top of GPS + Radar*? That baseline is trained in part 1, so part 2cannot compute its own deltas, and neither training notebook can produce thefinal table alone. Keeping the reporting here means there is exactly one placethe paper's numbers come from.## What to attachUpload the per-configuration JSONs from **both** parts as a Kaggle dataset andattach it. The search is recursive and matches on file content, not path, so anyfolder layout works.To run locally instead, point `SEARCH_ROOTS` at the directories holding thedownloaded JSONs.

## 1. Where to look

In [ ]:
import jsonimport osfrom pathlib import Pathimport pandas as pd# Searched recursively, in order. Missing directories are ignored, so the same# notebook works on Kaggle and on a laptop.SEARCH_ROOTS = [    Path("/kaggle/input"),                                  # attached datasets    Path("/kaggle/working/outputs/modality_ablation"),      # this session    Path("outputs/modality_ablation"),                      # local    Path("results/modality_ablation"),                      # local, committed]OUT = (Path("/kaggle/working/outputs/modality_ablation")       if Path("/kaggle/working").exists() else Path("outputs/modality_ablation"))(OUT / "plots").mkdir(parents=True, exist_ok=True)# The eight configurations, in the order the table should read.ORDER = ["gps", "gps_radar", "gps_image", "gps_lidar", "gps_radar_image",         "gps_radar_lidar", "gps_image_lidar", "gps_radar_image_lidar"]BASE = "gps_radar"print("search roots:")for r in SEARCH_ROOTS:    print(f"  {'OK     ' if r.exists() else 'absent '} {r}")print(f"\noutputs -> {OUT}")

## 2. Discover the per-configuration resultsDeduplicated by `slug`, so attaching overlapping datasets is harmless.

In [ ]:
def discover():    # every per-configuration record found, keyed by slug    found, sources = {}, {}    for root in SEARCH_ROOTS:        if not root.exists():            continue        for dirpath, _dirs, files in os.walk(root, followlinks=True):            for name in sorted(files):                if not name.endswith(".json"):                    continue                path = Path(dirpath) / name                try:                    rec = json.loads(path.read_text())                except Exception:                    continue                # a per-configuration record, not a config/summary file                if not all(k in rec for k in ("slug", "label", "metrics", "cost")):                    continue                if rec["slug"] not in found:                    found[rec["slug"]] = rec                    sources[rec["slug"]] = path    return found, sourcesrecords, sources = discover()print(f"{len(records)} configuration(s) found\n")for slug in ORDER:    if slug in records:        r = records[slug]        print(f"  [part {r.get('part', '?')}] {r['label']:<34} {sources[slug]}")    else:        print(f"  [ missing ] {slug}")missing = [s for s in ORDER if s not in records]if missing:    print(f"\n{len(missing)} missing: {', '.join(missing)}")    print("The tables below are partial. Attach the other part's per_config JSONs.")if not records:    raise SystemExit("No per-configuration results found. Attach the JSONs from the "                     "training notebooks, or point SEARCH_ROOTS at them.")

## 3. Protocol consistency checkA controlled ablation is only valid if every configuration was trained the sameway. This compares the recorded protocol across configurations and says soloudly if they disagree, rather than presenting a merged table as if they did.

In [ ]:
PROTOCOL_KEYS = ["epochs", "batch_size", "lr", "pool", "seed",                 "modality_dropout", "use_historical_beams"]protocol = pd.DataFrame([{"slug": r["slug"], **{k: r.get(k) for k in PROTOCOL_KEYS}}                         for r in records.values()])print(protocol.to_string(index=False), "\n")inconsistent = {k: sorted(set(protocol[k].dropna()))                for k in PROTOCOL_KEYS if protocol[k].nunique(dropna=True) > 1}if inconsistent:    print("WARNING -- these configurations were NOT trained under one protocol:")    for k, vals in inconsistent.items():        print(f"   {k}: {vals}")    print("\nDifferences in accuracy cannot be attributed to modality choice alone. "          "Retrain the odd ones out before reporting these numbers.")else:    print("protocol identical across all available configurations")EPOCHS_USED = int(protocol.epochs.iloc[0]) if "epochs" in protocol else None

## 4. Main results table

In [ ]:
def flatten(r, split="val"):    m = r["metrics"].get(split, {}).get("overall", {})    c = r["cost"]    return {        "Configuration": r["label"], "slug": r["slug"], "part": r.get("part"),        "Top-1": m.get("top1"), "Top-3": m.get("top3"),        "Top-5": m.get("top5"), "DBA": m.get("dba"), "n": m.get("n"),        "Params (M)": c["params_active"] / 1e6,        "Latency (ms)": c["latency_ms_per_sample"],        "GFLOPs": c["gflops_per_sample"],        "Train (min)": r["train_seconds"] / 60.0,        "Best epoch": r["best_epoch"],    }results = pd.DataFrame([flatten(records[s]) for s in ORDER if s in records])results.to_csv(OUT / "results.csv", index=False)print("VALIDATION split (2,198 samples, scenarios 32/33/34) -- primary result\n")print(results.drop(columns=["slug", "part"]).round(4).to_string(index=False))adaptation = pd.DataFrame([flatten(records[s], "adaptation")                           for s in ORDER if s in records])adaptation.to_csv(OUT / "results_adaptation.csv", index=False)print("\n\nADAPTATION split (100 samples, scenarios 31/32/33) -- secondary check\n")print(adaptation[["Configuration", "Top-1", "Top-3", "Top-5", "DBA", "n"]]      .round(4).to_string(index=False))

## 5. Incremental value over the cheap baselineThe number the adaptive-routing design actually needs.

In [ ]:
ADDED = {"image": "Camera", "lidar": "LiDAR", "radar": "Radar", "gps": "GPS"}deltas = pd.DataFrame()if BASE in records:    base = results[results.slug == BASE].iloc[0]    rows = []    for _, r in results.iterrows():        if r.slug == BASE:            continue        extra = [m for m in r.slug.split("_") if m not in BASE.split("_")]        if not extra:            continue        rows.append({            "Added to GPS + Radar": " + ".join(ADDED.get(a, a) for a in extra),            "Configuration": r["Configuration"],            "dTop-1": r["Top-1"] - base["Top-1"],            "dTop-3": r["Top-3"] - base["Top-3"],            "dDBA": r["DBA"] - base["DBA"],            "dParams (M)": r["Params (M)"] - base["Params (M)"],            "dGFLOPs": r["GFLOPs"] - base["GFLOPs"],            "dLatency (ms)": r["Latency (ms)"] - base["Latency (ms)"],        })    deltas = pd.DataFrame(rows).sort_values("dDBA", ascending=False)    deltas.to_csv(OUT / "deltas.csv", index=False)    print(f"baseline: {base['Configuration']}   Top-1 {base['Top-1']:.4f}   "          f"Top-3 {base['Top-3']:.4f}   DBA {base['DBA']:.4f}   "          f"{base['GFLOPs']:.3f} GFLOPs\n")    print(deltas.round(4).to_string(index=False))else:    print(f"'{BASE}' is missing -- it is trained in part 1, and the deltas need it.")

## 6. Scenario-wise resultsIs the usefulness of an expensive modality consistent across the fourenvironments? Scenarios 32/33/34 come from `val`; scenario 31 exists only in`adaptation` (n=50) and is correspondingly noisy.

In [ ]:
KEY = ["gps_radar", "gps_radar_image", "gps_radar_lidar", "gps_radar_image_lidar"]rows = []for slug in KEY:    if slug not in records:        continue    r = records[slug]    for split in ("val", "adaptation"):        for group, m in r["metrics"].get(split, {}).items():            if group == "overall" or not isinstance(m, dict):                continue            rows.append({"Configuration": r["label"], "slug": slug, "split": split,                         "scenario": group, "n": m["n"], "Top-1": m["top1"],                         "Top-3": m["top3"], "DBA": m["dba"]})scenario_results = pd.DataFrame(rows)if scenario_results.empty:    print("none of the key configurations are available yet")else:    scenario_results["_o"] = scenario_results.slug.apply(KEY.index)    scenario_results = (scenario_results.sort_values(["scenario", "_o"])                        .drop(columns="_o").reset_index(drop=True))    scenario_results.to_csv(OUT / "scenario_results.csv", index=False)    for scn, g in scenario_results.groupby("scenario"):        note = "   [adaptation only, n=50 -- noisy]" if scn == "scenario31" else ""        print(f"\n{scn}{note}")        print(g[["Configuration", "split", "n", "Top-1", "Top-3", "DBA"]]              .round(4).to_string(index=False))

## 7. Plots

In [ ]:
import matplotlib.pyplot as pltPLOTS = OUT / "plots"short = results.Configuration.str.replace(" + ", "+", regex=False)fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))for a, metric, colour in ((ax[0], "DBA", "tab:green"), (ax[1], "Top-1", "tab:blue")):    a.barh(short, results[metric], color=colour)    a.set(xlabel=metric, title=f"{metric} by modality configuration")    a.grid(alpha=.3, axis="x")    a.invert_yaxis()    for i, v in enumerate(results[metric]):        a.text(v, i, f" {v:.3f}", va="center", fontsize=9)plt.tight_layout()plt.savefig(PLOTS / "ablation_main.png", dpi=150)plt.show()if not deltas.empty:    fig, ax = plt.subplots(1, 2, figsize=(15, 4.4))    colours = ["tab:red" if v >= 0 else "tab:grey" for v in deltas["dDBA"]]    ax[0].barh(deltas["Added to GPS + Radar"], deltas["dDBA"], color=colours)    ax[0].axvline(0, color="k", lw=.8)    ax[0].set(xlabel="change in DBA vs GPS + Radar",              title="Incremental value over the cheap baseline")    ax[0].grid(alpha=.3, axis="x")    ax[0].invert_yaxis()    ax[1].scatter(results["GFLOPs"], results["DBA"], s=70, color="tab:purple")    for _, r in results.iterrows():        ax[1].annotate(r.Configuration.replace(" + ", "+"), (r["GFLOPs"], r["DBA"]),                       fontsize=8, xytext=(4, 3), textcoords="offset points")    ax[1].set(xlabel="forward GFLOPs per sample", ylabel="DBA",              title="Performance vs inference cost")    ax[1].grid(alpha=.3)    plt.tight_layout()    plt.savefig(PLOTS / "ablation_deltas_and_cost.png", dpi=150)    plt.show()print(f"plots -> {PLOTS}")

## 8. Reference: the saved AMBER baselineShown as context if a previous full AMBER run is available. It is **not**protocol-matched to this ablation — it trains longer and with historical beamsenabled — so it is a reference point, not a fair comparator.

In [ ]:
def find_amber_baseline():    # history.csv from a previous full AMBER run, if one is reachable    for root in SEARCH_ROOTS + [Path("results"), Path("/kaggle/working/runs")]:        if not root.exists():            continue        for dirpath, _dirs, files in os.walk(root, followlinks=True):            if "history.csv" in files and "modality_ablation" not in str(dirpath):                return Path(dirpath)    return Noneamber_dir = find_amber_baseline()if amber_dir is None:    print("no saved AMBER baseline run found -- skipping the reference row")else:    hist = pd.read_csv(amber_dir / "history.csv")    best = hist.loc[hist.val_top1.idxmax()]    cfg_path = amber_dir / "config.json"    tr = (json.loads(cfg_path.read_text()).get("train", {})          if cfg_path.exists() else {})    print(f"AMBER baseline from {amber_dir}")    print(f"  its protocol : epochs {tr.get('epochs', '?')}, "          f"modality_dropout {tr.get('modality_dropout', '?')}, historical beams on")    print(f"  this ablation: epochs {EPOCHS_USED}, "          f"modality_dropout 0.0, historical beams off")    print(f"  best val  Top-1 {best.val_top1:.4f}   Top-3 {best.val_top3:.4f}   "          f"DBA {best.val_dba:.4f}   at epoch {int(best.epoch)}")    if "gps_radar_image_lidar" in records:        full = results[results.slug == "gps_radar_image_lidar"].iloc[0]        print(f"\n  ablation full multimodal: Top-1 {full['Top-1']:.4f}   "              f"Top-3 {full['Top-3']:.4f}   DBA {full['DBA']:.4f}")        print("  Any gap follows from the protocol difference above, not from "              "modality choice. Do not read it as an ablation result.")

## 9. InterpretationGenerated from the measured results only — including when they show thatadaptive routing is *not* worth pursuing.

In [ ]:
def interpret(results, deltas, scenario_results):    L, have = [], set(results.slug)    if BASE not in have:        return ["GPS + Radar is missing, so the comparisons this experiment exists "                "to make cannot be drawn. Run part 1."]    base = results[results.slug == BASE].iloc[0]    get = lambda s: results[results.slug == s].iloc[0] if s in have else None    gps, full = get("gps"), get("gps_radar_image_lidar")    cam, lid = get("gps_radar_image"), get("gps_radar_lidar")    if gps is not None:        L.append(f"**Radar over GPS alone.** GPS-only reaches DBA {gps['DBA']:.4f} "                 f"(Top-1 {gps['Top-1']:.4f}); adding radar gives DBA "                 f"{base['DBA']:.4f} (Top-1 {base['Top-1']:.4f}), "                 f"{base['DBA'] - gps['DBA']:+.4f} DBA for "                 f"{base['GFLOPs'] - gps['GFLOPs']:+.3f} GFLOPs.")    if full is not None:        frac = base["DBA"] / full["DBA"] if full["DBA"] else float("nan")        gap = full["DBA"] - base["DBA"]        verb = "retains" if frac <= 1.0 else "already exceeds"        L.append(f"**Is GPS + Radar a strong cheap baseline?** It {verb} "                 f"{frac * 100:.1f}% of the full model's DBA using "                 f"{base['GFLOPs'] / full['GFLOPs'] * 100:.0f}% of its FLOPs and "                 f"{base['Params (M)'] / full['Params (M)'] * 100:.0f}% of its "                 f"active parameters. The full model is {gap:+.4f} DBA "                 + ("better." if gap > 0 else                    "WORSE -- at this training budget the extra modalities are not "                    "paying for themselves, reported as measured."))    if cam is not None and lid is not None:        dc, dl = cam["DBA"] - base["DBA"], lid["DBA"] - base["DBA"]        L.append(f"**Camera vs LiDAR on top of GPS + Radar.** Camera {dc:+.4f} DBA "                 f"for {cam['GFLOPs'] - base['GFLOPs']:+.3f} GFLOPs; LiDAR "                 f"{dl:+.4f} DBA for {lid['GFLOPs'] - base['GFLOPs']:+.3f} GFLOPs.")        if max(dc, dl) < 0.01:            L.append("  Both gains are under 0.01 DBA. On this evidence neither "                     "expensive modality is worth selectively acquiring, and a "                     "router over them would have little to gain. **This argues "                     "against pursuing adaptive routing as currently framed** -- "                     "the honest next step is to ask why the expensive modalities "                     "contribute so little, not to build the gate.")        elif abs(dc - dl) < 0.005:            L.append(f"  They are within 0.005 DBA of each other, so neither "                     f"dominates. Both stay as candidate escalation routes and the "                     f"choice should be made on cost: Camera {cam['GFLOPs']:.3f} vs "                     f"LiDAR {lid['GFLOPs']:.3f} GFLOPs.")        else:            big = ("Camera", dc, cam) if dc > dl else ("LiDAR", dl, lid)            small = ("LiDAR", dl) if dc > dl else ("Camera", dc)            per_flop = big[1] / max(big[2]["GFLOPs"] - base["GFLOPs"], 1e-9)            L.append(f"  {big[0]} is the clearly more valuable escalation "                     f"({big[1]:+.4f} vs {small[1]:+.4f} DBA), so the adaptive "                     f"experiment should prioritise acquiring {big[0]} first. Per "                     f"unit of compute it returns {per_flop:.4f} DBA/GFLOP.")    if full is not None and (cam is not None or lid is not None):        best_single = max([x for x in (cam, lid) if x is not None],                          key=lambda r: r["DBA"])        extra = full["DBA"] - best_single["DBA"]        L.append(f"**Is the fourth modality worth it?** "                 f"{best_single['Configuration']} to full multimodal adds "                 f"{extra:+.4f} DBA for "                 f"{full['GFLOPs'] - best_single['GFLOPs']:+.3f} GFLOPs. "                 + ("Marginal, which supports escalating to one expensive modality "                    "rather than all of them -- exactly what a router would do."                    if extra < 0.01 else                    "A real gain, so full escalation should stay in the routing "                    "policy's action space."))    if not scenario_results.empty:        piv = (scenario_results[scenario_results.split == "val"]               .pivot_table(index="scenario", columns="slug", values="DBA"))        if BASE in piv.columns:            per = {name: (piv[slug] - piv[BASE]).round(4).to_dict()                   for slug, name in (("gps_radar_image", "Camera"),                                      ("gps_radar_lidar", "LiDAR"))                   if slug in piv.columns}            if per:                bits = "; ".join(                    f"{n}: " + ", ".join(f"{s.replace('scenario', 'S')} {v:+.3f}"                                         for s, v in d.items())                    for n, d in per.items())                spread = max(max(d.values()) - min(d.values()) for d in per.values())                L.append(f"**Consistency across environments (val, change in DBA vs "                         f"GPS + Radar).** {bits}. Spread across scenarios is "                         f"{spread:.3f} DBA"                         + (", so the benefit is roughly uniform and a single global "                            "modality set may suffice -- which weakens the case for "                            "per-sample routing." if spread < 0.02 else                            ", so the benefit is environment-dependent, which is "                            "itself evidence that a fixed modality set is "                            "suboptimal and routing has something to exploit."))    L.append(f"**Caveats.** {len(results)}/8 configurations; {EPOCHS_USED} epochs "             "rather than to full convergence; historical beam indices disabled "             "(not a challenge input, and absent from the whole test split); "             "scenario 31 rests on 50 adaptation samples; scenario 34 supplies "             "radar for only ~40% of its samples, so radar configurations are "             "partly GPS-only there.")    return Llines = interpret(results, deltas, scenario_results)text = "\n\n".join(lines)print(text)(OUT / "interpretation.md").write_text(text)

## 10. Save outputs

In [ ]:
summary = {    "n_configurations": int(len(results)),    "missing": [s for s in ORDER if s not in records],    "protocol": protocol.to_dict(orient="records"),    "protocol_consistent": not bool(inconsistent),    "baseline": BASE,    "epochs": EPOCHS_USED,    "results_val": results.to_dict(orient="records"),    "deltas": deltas.to_dict(orient="records") if not deltas.empty else [],}(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))print(f"{OUT}:")for p in sorted(OUT.rglob("*")):    if p.is_file():        print(f"  {p.relative_to(OUT)}  ({p.stat().st_size / 1e3:.1f} KB)")